# Accessing Claude with API
In this notebook, you will learn the core request patterns used with the Anthropic Python SDK. The examples progress from a single prompt to stateful conversations, system prompts, streaming responses, and techniques for getting cleaner structured output.
What to look for: how the `messages` payload changes from section to section, and how small prompt or parameter changes can significantly affect the model's behavior and output format.

## Making a Request
This section introduces the smallest useful Claude API workflow: import the SDK, load your API key, create a client, define a model and token limit, send one user message, and read the returned text from the response object.
Key takeaway: a basic request only needs a model, a token budget, and a `messages` list containing at least one user turn.

In [ ]:
# Import Dependencies
import os
from dotenv import load_dotenv
from anthropic import Anthropic

In [ ]:
# Load Environment Variables
load_dotenv()

In [ ]:
# Create API Client
client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
model = "claude-opus-4-6"
max_tokens = 1000
msg = "What are the top 5 most discontinuous innovations in Agentic AI and what value creation opportunities do they present for startups?"

In [ ]:
# Make a Request
message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{"role": "user", "content": msg}]
)

In [ ]:
message.content[0].text

## Multi-Turn Conversations
This section shows how to preserve context across multiple requests. Instead of sending isolated prompts, you maintain a running `messages` list that stores both user and assistant turns, then send the entire history back with each follow-up question.
Key takeaway: conversation memory is not automatic in the API. You preserve context by explicitly appending prior turns and resubmitting them in the next request.

### Helper Functions
These helper functions reduce repetition and make the conversation flow easier to read. Two functions append user and assistant turns to the message history, while the `chat(...)` wrapper centralizes the API call and optionally accepts a system prompt or stop sequences.
Key takeaway: thin helper functions make it easier to experiment with prompting patterns without rewriting the request structure each time.

In [30]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})

def chat(messages, model="claude-sonnet-4-20250514", max_tokens=1000, system=None, stop_sequences=None):
    kwargs = dict(
        model=model,
        max_tokens=max_tokens,
        messages=messages,
    )
    if system is not None:
        kwargs["system"] = system
    if stop_sequences is not None:
        kwargs["stop_sequences"] = stop_sequences

    message = client.messages.create(**kwargs)
    return message.content[0].text

In [ ]:
# Make a starting message list
messages = []

# Add the initial user question
msg1 = "What is the market value of being able to build model-agnostic agentic AI systems from scratch, without frameworks, using only APIs?"
add_user_message(messages, msg1)

# Pass the list of messages to the chat function
answer = chat(messages)
print(answer)

# Take the answer and add it to the messages list
add_assistant_message(messages, answer)

In [ ]:
# Add a follow-up question to the user message list
msg2 = "What skills are most important for building model-agnostic agentic AI systems from scratch, without frameworks, using only APIs?"
add_user_message(messages, msg2)

# Call the chat function again with the updated messages list
answer2 = chat(messages)
print(answer2)

# Take the answer and add it to the messages list
add_assistant_message(messages, answer2)

In [ ]:
# Add a follow-up question to the user message list
msg3 = "My use case is that of an agentic startup founder, not an employee. How would you characterize the skill set for building agentic systems from first principles as a founder?"
add_user_message(messages, msg3)

# Call the chat function again with the updated messages list
answer3 = chat(messages)
print(answer3)

# Take the answer and add it to the messages list
add_assistant_message(messages, answer3)

## System Prompts
This section compares a normal request with a request shaped by system instructions. The user asks for the same kind of coding help in both cases, but the second call adds a system prompt to steer Claude toward concise, engineer-style output.
Key takeaway: system prompts are a reliable way to control style, tone, and constraints without changing the user's original request.

In [8]:
messages = []
add_user_message(messages, "Write a Python function that checks a string for duplicate characters and returns a list of the duplicates.")
answer = chat(messages)
print(answer)

## Function to Find Duplicate Characters in a String

```python
def find_duplicate_characters(string: str) -> list[str]:
    """
    Checks a string for duplicate characters and returns a list of duplicates.

    Args:
        string: The input string to check for duplicates.

    Returns:
        A list of characters that appear more than once in the string.

    Examples:
        >>> find_duplicate_characters("hello")
        ['l']
        >>> find_duplicate_characters("programming")
        ['r', 'g', 'm']
        >>> find_duplicate_characters("abc")
        []
    """
    if not string:
        return []

    char_count = {}
    for char in string:
        char_count[char] = char_count.get(char, 0) + 1

    return [char for char, count in char_count.items() if count > 1]
```

### Example Usage

```python
test_strings = [
    "hello",
    "programming",
    "abc",
    "",
    "aabbcc",
    "Python",
    "Mississippi",
]

for s in test_strings:
    duplicates = find_duplicate_charact

In [ ]:
system = [{"type": "text", "text":"You are a Python engineer who writes very concise code."}]
answer = chat(messages, model="claude-sonnet-4-6", max_tokens=1000, system=system)
print(answer)

```python
def find_duplicates(s):
    seen = set()
    return list({char for char in s if char in seen or seen.add(char)})
```

**Example usage:**
```python
print(find_duplicates("programming"))  # ['r', 'g', 'm']
print(find_duplicates("hello"))        # ['l']
print(find_duplicates("abcdef"))       # []
```

**How it works:**
- Uses a set comprehension with a `seen` set to track visited characters
- `char in seen` → duplicate found, include it
- `seen.add(char)` → returns `None` (falsy), so new chars are added to `seen` but excluded from the result
- Wraps in `list()` to return a list of unique duplicates


## Response Streaming
These examples show how to receive output incrementally rather than waiting for the entire response to finish. The first example exposes low-level streaming events, while the second uses the higher-level stream helper to iterate over text chunks and then access the final assembled message.
Key takeaway: streaming improves responsiveness and gives you more control when building real-time interfaces or monitoring generation as it happens.

In [ ]:
messages = []

add_user_message(messages, "Write a one-sentence description of a fake database.")

stream = client.messages.create(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=messages,
    stream=True)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01WEUMhJfavcE19wmr2kEc9K', container=None, content=[], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=1, server_tool_use=None, service_tier='standard')), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='"', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='EmployeeHub is a fictional corporate database containing records of 10', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text=',000 imaginary employees, includin

In [ ]:
messages = []

add_user_message(messages, "Write a one-sentence description of a fake database.")

with client.messages.stream(
    model="claude-sonnet-4-6",
    max_tokens=1000,
    messages=messages) as stream:
    for _text in stream.text_stream:
        pass
stream.get_final_message()

ParsedMessage(id='msg_01XobJpPynDBysCtFT7KJjbA', container=None, content=[ParsedTextBlock(citations=None, text='"The Galactic Registry of Sentient Spoons is a fictional database cataloging over 47 million self-aware utensils across the universe, including their names, preferred soups, and emotional backstories."', type='text', parsed_output=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=18, output_tokens=50, server_tool_use=None, service_tier='standard'))

## Structured Data
This section explores how to guide Claude toward cleaner machine-readable output. The first example asks for JSON directly, while the second uses a formatting trick: it starts the assistant response inside a JSON code fence and stops generation before the closing fence appears.
Key takeaway: when you need structured output, prompt wording helps, but shaping the response format with assistant turns and stop sequences can make the result much easier to parse.

In [ ]:
messages = []
msg = "Generate a very short EventBridge rule as JSON."
add_user_message(messages, msg)
answer = chat(messages)
print(answer)

In [ ]:
messages = []
add_user_message(messages, "Generate a very short EventBridge rule as JSON.")
add_assistant_message(messages, "```json")  # This nudges the model to continue directly with JSON content instead of adding extra explanation.

answer = chat(messages, stop_sequences=["```"])

print(answer)

/tmp/ipykernel_54502/3036598960.py:18: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**kwargs)



{
  "Name": "OrderProcessingRule",
  "EventPattern": {
    "source": ["myapp.orders"],
    "detail-type": ["Order Placed"]
  },
  "State": "ENABLED",
  "Targets": [
    {
      "Id": "1",
      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"
    }
  ]
}



## Structured Data Exercise
This exercise applies the same formatting technique to shell commands instead of JSON. The first cell asks for short AWS CLI commands in free-form text, and the second nudges Claude to return only bash code by beginning an assistant code fence and stopping before the closing delimiter.
Key takeaway: the same output-shaping pattern can be reused across formats such as JSON, Bash, SQL, or other structured text that you want to keep clean and predictable.

In [35]:
messages = []
prompt = """Generate three different sample AWS CLI commands. Each should be very short."""

add_user_message(messages, prompt)
text = chat(messages)
print(text)

/tmp/ipykernel_54502/3036598960.py:18: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**kwargs)


Here are three short AWS CLI commands:

1. **List S3 buckets:**
   ```bash
   aws s3 ls
   ```

2. **Describe EC2 instances:**
   ```bash
   aws ec2 describe-instances
   ```

3. **List IAM users:**
   ```bash
   aws iam list-users
   ```


In [36]:
add_assistant_message(messages, "```bash")
text = chat(messages, stop_sequences=["```"])
print(text)

/tmp/ipykernel_54502/3036598960.py:18: DeprecationWarning: The model 'claude-sonnet-4-20250514' is deprecated and will reach end-of-life on June 15th, 2026.
Please migrate to a newer model. Visit https://docs.anthropic.com/en/docs/resources/model-deprecations for more information.
  message = client.messages.create(**kwargs)



aws s3 ls

aws ec2 describe-instances

aws iam list-users

